# Pipeline: Gestión Semanal de Términos Orgánicos del Buscador

## Descripción general

Este notebook orquesta el pipeline completo de actualización semanal de los listados de productos
para el buscador de un e-commerce de gran escala.

El objetivo es determinar qué productos mostrar (y en qué orden) para cada término de búsqueda,
combinando datos de comportamiento del usuario con criterio analítico del equipo.

### Flujo del pipeline

```
[Diccionario de términos]  [Buscador + Ventas]  [Apuestas de especialistas]
         │                        │                        │
         ▼                        ▼                        │
  1. Sincronización          2. Cálculo de            3. Cruce de
     del dashboard              performance              listados
     (PreProcesos)              (CalculosPerformance)    (GeneradorListados)
                                        │                        │
                                        ▼                        │
                               Clasificación Si/No               │
                               (Manual o Modelo RF)              │
                                        │                        │
                                        └────────────────────────┘
                                                     │
                                                     ▼
                                        4. Actualización en el
                                           motor de búsqueda
                                           (ApisConnections)
```

### Módulos utilizados
- `preprocesos.py` — Sincronización de datos y gestión del histórico de decisiones
- `calculosperformance.py` — Score de performance por término-SKU
- `listados.py` — Generación y mezcla de listados finales
- `conexionesapis.py` — Conexión y actualización en el motor de búsqueda

## 1. Configuración: librerías, módulos y variables del pipeline

In [ ]:
import sys
from google.colab import drive
import pandas as pd

# Montar Google Drive para acceder a los archivos del equipo
drive.mount('/content/drive')

# Agregar la carpeta del proyecto al path para importar los módulos
sys.path.append('/content/drive/Shareddrives/[TEAM]/[PROJECT]/Scripts/')

from conexionesapis import ApisConnections
from preprocesos import PreProcesos
from calculosperformance import CalculosPerformance
from listados import GeneradorListados

In [ ]:
# =============================================================================
# VARIABLES DE CONFIGURACIÓN DEL PIPELINE
# Centralizar todas las variables aquí facilita la adaptación semanal
# sin tener que modificar el código de los módulos.
# =============================================================================

# --- Google Sheets: Diccionario maestro de términos ---
SPREADSHEET_DICCIONARIO = '[NOMBRE_SHEET_DICCIONARIO]'   # Sheet con el diccionario de términos
SHEET_DICCIONARIO = '[NOMBRE_HOJA]'                      # Pestaña con los datos
COL_NAME_DICCIONARIO = 'Término'                         # Columna con el nombre del término
COL_VAL_DICCIONARIO = 'Val'                              # Columna con el estado (ok/inactivo)
VALID_DICCIONARIO = 'ok'                                 # Valor que indica término activo

# --- Google Sheets: Dashboard de seguimiento ---
SPREADSHEET_DASHBOARD = '[NOMBRE_SHEET_DASHBOARD]'
SHEET_DASHBOARD = '[NOMBRE_HOJA]'

# --- Columnas del diccionario a conservar en el pipeline ---
COLS_A_CONS_DICT = [
    'Analista', 'Termino Representativo', 'ID', 'name_rule', 'Término',
    'tipo_carga', 'n_performance', 'n_especialistas', 'listado_especial',
    'Espacios_especiales', 'cruces_1p_3p', 'cruce_1p', 'cruce_3p'
]

# --- Rutas en Drive para el histórico de decisiones ---
HISTORICO_DE_DECISIONES = '/content/drive/Shareddrives/[TEAM]/[PROJECT]/compilado_decisiones.csv'
RUTA_FOLDER_TRABAJO = '/content/drive/Shareddrives/[TEAM]/[PROJECT]/Calculo_semanal/'

# --- Configuración del histórico de decisiones ---
COLS_A_MANTENER_DEL_DF_ANTERIOR = ['termino_agg', 'sku', 'des_articulo', 'decision_final']
COLS_PIVOTE_PARA_QUITAR_DUPLICADOS = ['sku', 'termino_agg']
DICT_PARA_RENOMBRAR_COLUMNAS = {'ter_agg': 'termino_agg'}
COL_DECISION_FINAL = 'decision_final'

# --- Configuración especial para el modo de especialistas (sku_rank) ---
COLS_AGRUPACION_ESPECIALISTAS = [
    'termino_agg', 'sku', 'rank', 'des_articulo', 'des_area',
    'des_categoria', 'des_marca', 'imp_precioventa', 'imp_preciodescuento',
    'por_descuento', 'por_cobertura', 'num_existenciatopoe', 'num_existencia'
]
DICT_RENOMBRES_ESPECIALISTAS = {'term representativo': 'termino_agg'}
COLS_ORDEN_ESPECIALISTAS = ['termino_agg', 'rank']
ORDEN_ASC_ESPECIALISTAS = [True, False]

# --- Rutas de los modelos de clasificación automática ---
PATH_DEL_MODELO = '/content/drive/Shareddrives/[TEAM]/[PROJECT]/Scripts/modelo_clasificador_articulos.pkl'
PATH_DE_VECTORIZADOR = '/content/drive/Shareddrives/[TEAM]/[PROJECT]/Scripts/vectorizador_tfidf.pkl'

# --- Configuración del motor de búsqueda ---
SPREADSHEET_APUESTAS = '[NOMBRE_SHEET_APUESTAS]'
TAG_LW = '[TAG_REGLAS]'  # Tag que identifica el tipo de reglas a actualizar

## 2. Preproceso: sincronización del diccionario y actualización del histórico

En este bloque se realizan dos tareas de mantenimiento de datos:

1. **Sincronización del dashboard**: Detecta si se agregaron o eliminaron términos
   en el diccionario maestro y actualiza el dashboard de seguimiento para que ambos
   sistemas siempre trabajen con el mismo conjunto de términos.

2. **Actualización del histórico de decisiones**: Incorpora las decisiones
   (Si/No por SKU-término) del archivo de la semana anterior al CSV acumulado,
   que sirve como memoria del modelo de clasificación automática.

In [ ]:
# Instanciar PreProcesos (realiza la autenticación OAuth de Google automáticamente)
sm = PreProcesos()

In [ ]:
# Sincronizar términos entre diccionario maestro y dashboard
# Detecta altas/bajas de términos y actualiza el dashboard si hay cambios
sm.sync_variants(
    spreadsheet_dict=SPREADSHEET_DICCIONARIO,
    sheet_dict=SHEET_DICCIONARIO,
    spreadsheet_dash=SPREADSHEET_DASHBOARD,
    sheet_dash=SHEET_DASHBOARD,
    col_name=COL_NAME_DICCIONARIO,
    col_val=COL_VAL_DICCIONARIO,
    valid=VALID_DICCIONARIO
)

In [ ]:
# Cargar el diccionario con los parámetros de configuración por término
df_dic = sm.hist_dict(
    spreadsheet_dict=SPREADSHEET_DICCIONARIO,
    sheet_dict=SHEET_DICCIONARIO,
    cols=COLS_A_CONS_DICT,
    col_val=COL_VAL_DICCIONARIO,
    valid=VALID_DICCIONARIO,
    col_ter=COL_NAME_DICCIONARIO
)

In [ ]:
# Actualizar el historial acumulado de decisiones con el archivo de la semana anterior
# El método detecta automáticamente el Excel más reciente en la carpeta de trabajo
# y pide confirmación antes de incorporarlo
df_decisiones = sm.update_historico_decisiones(
    path_csv_hist=HISTORICO_DE_DECISIONES,
    folder_new_files=RUTA_FOLDER_TRABAJO,
    cols_to_keep=COLS_A_MANTENER_DEL_DF_ANTERIOR,
    subset_dupes=COLS_PIVOTE_PARA_QUITAR_DUPLICADOS,
    rename_cols=DICT_PARA_RENOMBRAR_COLUMNAS,
    col_to_standardize=COL_DECISION_FINAL
)

## 3. Cálculo de performance y clasificación de artículos

Se calcula un score de relevancia por par (término, SKU) combinando:
- **Vistas** al detalle del producto desde el buscador
- **Agregados a carrito** desde el buscador
- **Tasa de venta/carrito** del SKU en general
- **Ingresos** y **unidades** generadas

Los SKUs sin historial de decisión se etiquetan como 'Revisar' y se clasifican
de forma manual o automática con el modelo Random Forest entrenado.

In [ ]:
# Cargar los archivos de datos del período actual
# Estos archivos contienen: términos buscados, datos de catálogo y métricas de ventas
df_buscadorv4 = pd.read_csv(RUTA_FOLDER_TRABAJO + 'buscador_terminos_filtrados.csv')
df_acomodos = pd.read_csv(RUTA_FOLDER_TRABAJO + 'acomodo_articulos_niveles.csv')
df_mkp = pd.read_csv(RUTA_FOLDER_TRABAJO + 'visitas_conversion_marketplace.csv')
df_onep = pd.read_csv(RUTA_FOLDER_TRABAJO + 'visitas_conversion_ecom.csv')

In [ ]:
# Inicializar la clase de cálculo de performance con los modelos de clasificación
cp = CalculosPerformance(
    path_modelo=PATH_DEL_MODELO,
    path_tfidf=PATH_DE_VECTORIZADOR,
    carpeta_salida=RUTA_FOLDER_TRABAJO
)

In [ ]:
# Calcular el score de performance para el flujo estándar de términos orgánicos
#
# Pesos configurables según la estrategia comercial de la semana:
#   w_vistas + w_carrito + w_ingreso + w_unidades + w_tasa = 1.0
#
# Configuración actual: mayor peso en tasa de conversión (venta/carrito)
df_resultado_performance = cp.calcular_performance(
    df_buscador=df_buscadorv4,
    df_domo=df_acomodos,
    df_dict=df_dic,
    df_decisiones=df_decisiones,
    mkp=df_mkp,
    onep=df_onep,
    w_vistas=0.23,
    w_carrito=0.23,
    w_ingreso=0.02,
    w_unidades=0.02,
    w_tasa=0.50,
    modo_cruce='termino'
)

# Gestionar los SKUs pendientes de clasificación:
#   Opción 1: Exportar a Excel para revisión manual del analista
#   Opción 2: Clasificar automáticamente con el modelo Random Forest
df_clasificado = cp.gestionar_clasificacion(df_resultado_performance)

In [ ]:
# Si se eligió revisión manual (opción 1), el proceso se pausó aquí.
# Después de editar el Excel en Drive, reanudar desde esta celda:
# df_clasificado = cp.cargar_excel_revisado('[NOMBRE_ARCHIVO_REVISADO].xlsx')

## 4. Generación del cruce de listados

Se combina el resultado del cálculo de performance con las **apuestas de especialistas**
(selección manual semanal del equipo analítico) para generar el listado final por término.

La mezcla respeta:
- Proporción configurable de SKUs propios (1P) vs marketplace (3P)
- SKUs prioritarios con posición fija al inicio del listado
- Número de SKUs de cada fuente a intercalar por ciclo

In [ ]:
# Cargar el archivo clasificado más reciente (si se pausó en el paso anterior)
ultimo_archivo = sm._get_latest_file(folder_path=cp.carpeta_salida)

if ultimo_archivo:
    df_final = cp.cargar_excel_revisado(ultimo_archivo)
    print(f"Archivo cargado: {ultimo_archivo}")
else:
    print("No se encontró ningún archivo. Verifica la carpeta de salida.")

In [ ]:
# Normalizar columnas del diccionario y generar el cruce final de listados
df_dic.columns = df_dic.columns.str.lower().str.strip()
ls = GeneradorListados(gspread_client=sm.client, df_dict=df_dic)

df_cruce_final = ls.cruce_de_listados(
    spreadsheet_apuestas=SPREADSHEET_APUESTAS,
    df_performance=df_final,
    col_decision_final=COL_DECISION_FINAL,
    c_semanas=1  # Leer las apuestas de la semana anterior
)

# Exportar el cruce final para revisión antes de cargar al motor de búsqueda
cp.exportar_excel(df_cruce_final, prefijo='Cruce_de_listados')

## 5. Actualización en el motor de búsqueda

Con el listado final validado, se actualiza el motor de búsqueda mediante su API REST.

El proceso incluye:
1. Autenticación con credenciales del analista
2. Validaciones previas (IDs duplicados, IDs inexistentes en producción)
3. Actualización local con recorte de SKUs al límite del motor
4. Auditoría de integridad del JSON antes de subir
5. Carga a producción con reporte de resultados
6. Validación post-carga comparando producción vs local

In [ ]:
# Inicializar la conexión con el motor de búsqueda
# IMPORTANTE: Las credenciales se ingresan de forma interactiva, nunca se almacenan
api = ApisConnections()
api.authenticate_lw()

# Eliminar duplicados por ID antes de procesar (precaución por posibles merges previos)
df_cruce_limpio = df_cruce_final.drop_duplicates(subset=['id_x'], keep='first')
print(f"Reglas a actualizar: {len(df_cruce_limpio)}")

In [ ]:
# Preparar los cambios localmente con validaciones y recorte de SKUs
# Este paso NO modifica producción todavía
df_listo_para_subir = api.actualizar_reglas_localmente(
    df_nuevos_listados=df_cruce_limpio,
    col_id_match='id_x',
    col_listado_nuevo='listado_final',
    tag=TAG_LW
)

In [ ]:
# Auditoría de integridad local: verificar que los listados procesados
# coincidan con los listados calculados antes de subir a producción
#
# Las discrepancias esperadas son las reglas que superaron el límite de SKUs
# y fueron recortadas a 900. El resto debe coincidir exactamente.

def obtener_skus_de_params(params_list):
    """Extrae el valor de elevateIds de la estructura de params de Lucidworks."""
    if isinstance(params_list, list):
        for item in params_list:
            if isinstance(item, dict) and item.get('key') == 'elevateIds':
                return item.get('value')
    return None

df_auditoria = df_cruce_limpio[['id_x', 'listado_final']].merge(
    df_listo_para_subir[['id', 'params']],
    left_on='id_x', right_on='id', how='inner'
)
df_auditoria['listado_procesado'] = df_auditoria['params'].apply(obtener_skus_de_params)
df_auditoria['check_ok'] = df_auditoria['listado_final'] == df_auditoria['listado_procesado']

total = len(df_auditoria)
exitosos = df_auditoria['check_ok'].sum()
recortadas = df_auditoria[~df_auditoria['check_ok']]

print(f"Total de reglas: {total}")
print(f"Coincidencias exactas: {exitosos}")
if not recortadas.empty:
    print(f"Reglas recortadas por límite de SKUs: {len(recortadas)}")
    print("(Normal si alguna regla superaba el límite máximo del motor)")
else:
    print("Todo correcto. Se puede proceder con la carga.")

In [ ]:
# Cargar los cambios a producción
# Genera un reporte detallado con el status HTTP de cada regla actualizada
df_reporte_carga = api.upload_changes(df_listo_para_subir)